# BirdlyQuest · bird photo ID model

This notebook trains the small on-phone model that powers **✨ Identify it** in BirdlyQuest.

**How to run it**
1. **Runtime → Change runtime type → T4 GPU**, then **Save**.
2. **Runtime → Run all**. Say *yes* when it asks to connect Google Drive. Progress is saved there, so if Colab disconnects you can just **Run all** again and it picks up where it stopped.
3. Wait about **3–5 hours**. You can close the tab, but keep the browser running if you can.
4. At the end a file called **`birdlyquest-birdid.zip`** downloads (it is also saved in your Drive under `BirdlyQuest-birdid/`). Send that zip back to Claude.

**What it does (and why it is safe to sell)**
* It picks about 800 species: your 122 BirdDex birds plus the most-seen birds in each of the app's regions.
* It downloads only iNaturalist photos licensed **CC0** (public domain) or **CC BY** (free to use commercially with credit). Photos with a *non-commercial* licence are never downloaded.
* It records every photographer, so the app can show the credits that CC BY requires.
* It fine-tunes a small, phone-friendly image model (EfficientNet-B0) and exports it for the browser.

In [ ]:
# ① Setup ───────────────────────────────────────────────────────────────────────────
!pip -q install timm onnx onnxruntime
import os, json, time, random, re, csv, math, shutil, zipfile, io, concurrent.futures as cf
import requests, torch
from PIL import Image
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE: switch Runtime → Change runtime type → T4 GPU, then Run all again')
assert torch.cuda.is_available(), 'No GPU. Runtime → Change runtime type → T4 GPU'

from google.colab import drive
drive.mount('/content/drive')
WORK = '/content/drive/MyDrive/BirdlyQuest-birdid'
os.makedirs(WORK, exist_ok=True)
FAST = '/content/data'                      # images are copied here for fast training
print('Saving progress to', WORK)

In [ ]:
# ② Settings ────────────────────────────────────────────────────────────────────────
TOP_PER_REGION = 150        # most-seen species per region
PHOTOS_PER_SPECIES = 120    # max training photos per species (one per observation)
MIN_PHOTOS = 25             # species with fewer usable photos are left out
LICENSES = ['cc0', 'cc-by'] # ONLY commercially usable licences. Do not add cc-by-nc or cc-by-sa.
EPOCHS = 14
IMG = 224
SEED = 7
UA = {'User-Agent': 'BirdlyQuest-birdid-training (github.com/ggranger757/BirdlyQuest)'}

# The app's BirdDex species and regions (generated from the app's code)
APP_SPECIES = [{"id": "amerob", "common": "American Robin", "sci": "Turdus migratorius"}, {"id": "norcar", "common": "Northern Cardinal", "sci": "Cardinalis cardinalis"}, {"id": "blujay", "common": "Blue Jay", "sci": "Cyanocitta cristata"}, {"id": "amecro", "common": "American Crow", "sci": "Corvus brachyrhynchos"}, {"id": "moudov", "common": "Mourning Dove", "sci": "Zenaida macroura"}, {"id": "houspa", "common": "House Sparrow", "sci": "Passer domesticus"}, {"id": "eursta", "common": "European Starling", "sci": "Sturnus vulgaris"}, {"id": "rocpig", "common": "Rock Pigeon", "sci": "Columba livia"}, {"id": "amegfi", "common": "American Goldfinch", "sci": "Spinus tristis"}, {"id": "dowwoo", "common": "Downy Woodpecker", "sci": "Dryobates pubescens"}, {"id": "bkcchi", "common": "Black-capped Chickadee", "sci": "Poecile atricapillus"}, {"id": "mallar", "common": "Mallard", "sci": "Anas platyrhynchos"}, {"id": "cangoo", "common": "Canada Goose", "sci": "Branta canadensis"}, {"id": "rewbla", "common": "Red-winged Blackbird", "sci": "Agelaius phoeniceus"}, {"id": "comloo", "common": "Common Loon", "sci": "Gavia immer"}, {"id": "daejun", "common": "Dark-eyed Junco", "sci": "Junco hyemalis"}, {"id": "houfin", "common": "House Finch", "sci": "Haemorhous mexicanus"}, {"id": "whbnut", "common": "White-breasted Nuthatch", "sci": "Sitta carolinensis"}, {"id": "tuftit", "common": "Tufted Titmouse", "sci": "Baeolophus bicolor"}, {"id": "sonspa", "common": "Song Sparrow", "sci": "Melospiza melodia"}, {"id": "comgra", "common": "Common Grackle", "sci": "Quiscalus quiscula"}, {"id": "balori", "common": "Baltimore Oriole", "sci": "Icterus galbula"}, {"id": "cedwax", "common": "Cedar Waxwing", "sci": "Bombycilla cedrorum"}, {"id": "rethaw", "common": "Red-tailed Hawk", "sci": "Buteo jamaicensis"}, {"id": "coohaw", "common": "Cooper's Hawk", "sci": "Astur cooperii"}, {"id": "baleag", "common": "Bald Eagle", "sci": "Haliaeetus leucocephalus"}, {"id": "osprey", "common": "Osprey", "sci": "Pandion haliaetus"}, {"id": "amekes", "common": "American Kestrel", "sci": "Falco sparverius"}, {"id": "grhowl", "common": "Great Horned Owl", "sci": "Bubo virginianus"}, {"id": "easowl", "common": "Eastern Screech-Owl", "sci": "Megascops asio"}, {"id": "snoowl", "common": "Snowy Owl", "sci": "Bubo scandiacus"}, {"id": "grbher", "common": "Great Blue Heron", "sci": "Ardea herodias"}, {"id": "killde", "common": "Killdeer", "sci": "Charadrius vociferus"}, {"id": "ribgul", "common": "Ring-billed Gull", "sci": "Larus delawarensis"}, {"id": "rthhum", "common": "Ruby-throated Hummingbird", "sci": "Archilochus colubris"}, {"id": "yelwar", "common": "Yellow Warbler", "sci": "Setophaga petechia"}, {"id": "yerwar", "common": "Yellow-rumped Warbler", "sci": "Setophaga coronata"}, {"id": "comyel", "common": "Common Yellowthroat", "sci": "Geothlypis trichas"}, {"id": "amered", "common": "American Redstart", "sci": "Setophaga ruticilla"}, {"id": "bawwar", "common": "Black-and-white Warbler", "sci": "Mniotilta varia"}, {"id": "ovenbi", "common": "Ovenbird", "sci": "Seiurus aurocapilla"}, {"id": "norfli", "common": "Northern Flicker", "sci": "Colaptes auratus"}, {"id": "pilwoo", "common": "Pileated Woodpecker", "sci": "Dryocopus pileatus"}, {"id": "wooduc", "common": "Wood Duck", "sci": "Aix sponsa"}, {"id": "eurbla", "common": "Common Blackbird", "sci": "Turdus merula"}, {"id": "blutit", "common": "Eurasian Blue Tit", "sci": "Cyanistes caeruleus"}, {"id": "gretit", "common": "Great Tit", "sci": "Parus major"}, {"id": "eurrob", "common": "European Robin", "sci": "Erithacus rubecula"}, {"id": "eurmag", "common": "Eurasian Magpie", "sci": "Pica pica"}, {"id": "carcro", "common": "Carrion Crow", "sci": "Corvus corone"}, {"id": "hoocro", "common": "Hooded Crow", "sci": "Corvus cornix"}, {"id": "eurjay", "common": "Eurasian Jay", "sci": "Garrulus glandarius"}, {"id": "cowpig", "common": "Common Wood Pigeon", "sci": "Columba palumbus"}, {"id": "comcha", "common": "Common Chaffinch", "sci": "Fringilla coelebs"}, {"id": "eurgol", "common": "European Goldfinch", "sci": "Carduelis carduelis"}, {"id": "eurgre", "common": "European Greenfinch", "sci": "Chloris chloris"}, {"id": "bkhgul", "common": "Black-headed Gull", "sci": "Chroicocephalus ridibundus"}, {"id": "greher", "common": "Grey Heron", "sci": "Ardea cinerea"}, {"id": "tawowl", "common": "Tawny Owl", "sci": "Strix aluco"}, {"id": "eurwre", "common": "Eurasian Wren", "sci": "Troglodytes troglodytes"}, {"id": "comkes", "common": "Common Kestrel", "sci": "Falco tinnunculus"}, {"id": "grswoo", "common": "Great Spotted Woodpecker", "sci": "Dendrocopos major"}, {"id": "lottit", "common": "Long-tailed Tit", "sci": "Aegithalos caudatus"}, {"id": "comchi", "common": "Common Chiffchaff", "sci": "Phylloscopus collybita"}, {"id": "wlwwar", "common": "Willow Warbler", "sci": "Phylloscopus trochilus"}, {"id": "blackc", "common": "Eurasian Blackcap", "sci": "Sylvia atricapilla"}, {"id": "comnig", "common": "Common Nightingale", "sci": "Luscinia megarhynchos"}, {"id": "norgos", "common": "Eurasian Goshawk", "sci": "Astur gentilis"}, {"id": "egygoo", "common": "Egyptian Goose", "sci": "Alopochen aegyptiaca"}, {"id": "roripa", "common": "Rose-ringed Parakeet", "sci": "Psittacula krameri"}, {"id": "railor", "common": "Rainbow Lorikeet", "sci": "Trichoglossus moluccanus"}, {"id": "succoc", "common": "Sulphur-crested Cockatoo", "sci": "Cacatua galerita"}, {"id": "galah", "common": "Galah", "sci": "Eolophus roseicapilla"}, {"id": "ausmag", "common": "Australian Magpie", "sci": "Gymnorhina tibicen"}, {"id": "noimin", "common": "Noisy Miner", "sci": "Manorina melanocephala"}, {"id": "auwibi", "common": "Australian White Ibis", "sci": "Threskiornis molucca"}, {"id": "maglar", "common": "Magpie-lark", "sci": "Grallina cyanoleuca"}, {"id": "laukoo", "common": "Laughing Kookaburra", "sci": "Dacelo novaeguineae"}, {"id": "wilwag", "common": "Willie Wagtail", "sci": "Rhipidura leucophrys"}, {"id": "supfai", "common": "Superb Fairywren", "sci": "Malurus cyaneus"}, {"id": "crepig", "common": "Crested Pigeon", "sci": "Ocyphaps lophotes"}, {"id": "silgul", "common": "Silver Gull", "sci": "Chroicocephalus novaehollandiae"}, {"id": "commyn", "common": "Common Myna", "sci": "Acridotheres tristis"}, {"id": "houcro", "common": "House Crow", "sci": "Corvus splendens"}, {"id": "revbul", "common": "Red-vented Bulbul", "sci": "Pycnonotus cafer"}, {"id": "blkkit", "common": "Black Kite", "sci": "Milvus migrans"}, {"id": "categr", "common": "Cattle Egret", "sci": "Ardea ibis"}, {"id": "inpher", "common": "Indian Pond Heron", "sci": "Ardeola grayii"}, {"id": "pursun", "common": "Purple Sunbird", "sci": "Cinnyris asiaticus"}, {"id": "copbar", "common": "Coppersmith Barbet", "sci": "Psilopogon haemacephalus"}, {"id": "asikoe", "common": "Asian Koel", "sci": "Eudynamys scolopaceus"}, {"id": "bladro", "common": "Black Drongo", "sci": "Dicrurus macrocercus"}, {"id": "brebul", "common": "Brown-eared Bulbul", "sci": "Hypsipetes amaurotis"}, {"id": "warwhe", "common": "Warbling White-eye", "sci": "Zosterops japonicus"}, {"id": "eutspa", "common": "Eurasian Tree Sparrow", "sci": "Passer montanus"}, {"id": "labcro", "common": "Large-billed Crow", "sci": "Corvus macrorhynchos"}, {"id": "japtit", "common": "Japanese Tit", "sci": "Parus minor"}, {"id": "ortdov", "common": "Oriental Turtle Dove", "sci": "Streptopelia orientalis"}, {"id": "spbduc", "common": "Eastern Spot-billed Duck", "sci": "Anas zonorhyncha"}, {"id": "whiwag", "common": "White Wagtail", "sci": "Motacilla alba"}, {"id": "grekis", "common": "Great Kiskadee", "sci": "Pitangus sulphuratus"}, {"id": "rufhor", "common": "Rufous Hornero", "sci": "Furnarius rufus"}, {"id": "saffin", "common": "Saffron Finch", "sci": "Sicalis flaveola"}, {"id": "saytan", "common": "Sayaca Tanager", "sci": "Thraupis sayaca"}, {"id": "paltan", "common": "Palm Tanager", "sci": "Thraupis palmarum"}, {"id": "soulap", "common": "Southern Lapwing", "sci": "Vanellus chilensis"}, {"id": "trokin", "common": "Tropical Kingbird", "sci": "Tyrannus melancholicus"}, {"id": "banana", "common": "Bananaquit", "sci": "Coereba flaveola"}, {"id": "soucar", "common": "Southern Caracara", "sci": "Caracara plancus"}, {"id": "picpig", "common": "Picazuro Pigeon", "sci": "Patagioenas picazuro"}, {"id": "eardov", "common": "Eared Dove", "sci": "Zenaida auriculata"}, {"id": "shicow", "common": "Shiny Cowbird", "sci": "Molothrus bonariensis"}, {"id": "capspa", "common": "Cape Sparrow", "sci": "Passer melanurus"}, {"id": "smaswe", "common": "Southern Masked Weaver", "sci": "Ploceus velatus"}, {"id": "laudov", "common": "Laughing Dove", "sci": "Spilopelia senegalensis"}, {"id": "reedov", "common": "Red-eyed Dove", "sci": "Streptopelia semitorquata"}, {"id": "cagsta", "common": "Cape Starling", "sci": "Lamprotornis nitens"}, {"id": "hadibi", "common": "Hadada Ibis", "sci": "Bostrychia hagedash"}, {"id": "helgui", "common": "Helmeted Guineafowl", "sci": "Numida meleagris"}, {"id": "afrhoo", "common": "African Hoopoe", "sci": "Upupa africana"}, {"id": "capbul", "common": "Cape Bulbul", "sci": "Pycnonotus capensis"}, {"id": "sdcsun", "common": "Southern Double-collared Sunbird", "sci": "Cinnyris chalybeus"}]
APP_REGIONS = [{"code": "US-IL", "name": "Illinois", "lat": 40, "lng": -89.2}, {"code": "CA-ON", "name": "Ontario", "lat": 44.5, "lng": -79.5}, {"code": "GB", "name": "Great Britain", "lat": 52.8, "lng": -1.6}, {"code": "DE-BE", "name": "Berlin", "lat": 52.52, "lng": 13.4}, {"code": "AU-NSW", "name": "New South Wales", "lat": -33.3, "lng": 149}, {"code": "IN-DL", "name": "Delhi", "lat": 28.65, "lng": 77.2}, {"code": "JP-13", "name": "Tokyo", "lat": 35.68, "lng": 139.7}, {"code": "BR-SP", "name": "São Paulo", "lat": -23, "lng": -47.5}, {"code": "ZA-WC", "name": "Western Cape", "lat": -33.5, "lng": 19.5}]
print(len(APP_SPECIES), 'BirdDex species,', len(APP_REGIONS), 'regions')

In [ ]:
# ③ iNaturalist API helper: polite (about 1 request a second) and retries on errors ─────────────
API = 'https://api.inaturalist.org/v1'
_last = [0.0]
def api(path, **params):
    for attempt in range(6):
        wait = 1.05 - (time.time() - _last[0])
        if wait > 0: time.sleep(wait)
        _last[0] = time.time()
        try:
            r = requests.get(f'{API}/{path}', params=params, headers=UA, timeout=60)
            if r.status_code == 200: return r.json()
            if r.status_code in (429, 500, 502, 503, 504): time.sleep(10 * (attempt + 1)); continue
            r.raise_for_status()
        except requests.RequestException as e:
            print('  retrying after', e); time.sleep(10 * (attempt + 1))
    raise RuntimeError(f'iNaturalist API kept failing for {path}')

def load(name, default=None):
    p = os.path.join(WORK, name)
    return json.load(open(p)) if os.path.exists(p) else default
def save(name, obj):
    json.dump(obj, open(os.path.join(WORK, name), 'w'), ensure_ascii=False)

In [ ]:
# ④ Pick the species ────────────────────────────────────────────────────────────────
species = load('species.json')
if species is None:
    species = {}   # iNat taxon id → {sci, common, app_id}
    print('BirdDex birds…')
    for s in APP_SPECIES:
        j = api('taxa', q=s['sci'], rank='species', taxon_id=3, per_page=10)
        hit = next((t for t in j['results'] if t['name'].lower() == s['sci'].lower()), None)
        if not hit: print('  not found on iNaturalist:', s['sci']); continue
        species[str(hit['id'])] = {'sci': hit['name'], 'common': s['common'], 'app_id': s['id']}
    for r in APP_REGIONS:
        print('Most-seen birds in', r['name'], '…')
        dy, dx = 1.5, 2.0
        j = api('observations/species_counts', taxon_id=3, quality_grade='research', per_page=TOP_PER_REGION,
                nelat=r['lat'] + dy, nelng=r['lng'] + dx, swlat=r['lat'] - dy, swlng=r['lng'] - dx)
        for row in j['results']:
            t = row['taxon']
            if t.get('rank') != 'species': continue
            species.setdefault(str(t['id']), {'sci': t['name'], 'common': t.get('preferred_common_name') or t['name'], 'app_id': None})
    save('species.json', species)
print(len(species), 'species to learn')

In [ ]:
# ⑤ Find commercially usable photos (CC0 / CC BY only) ───────────────────────────────
photos = load('photos.json', {})      # taxon id → [{id, url, attribution, license, obs, user}]
todo = [t for t in species if t not in photos]
print(len(todo), 'species left to search')
for n, tid in enumerate(todo):
    got, seen_obs, page = [], set(), 1
    while len(got) < PHOTOS_PER_SPECIES and page <= 4:
        j = api('observations', taxon_id=tid, quality_grade='research', photos='true', photo_license=','.join(LICENSES),
                per_page=200, page=page, order_by='votes')
        if not j['results']: break
        for o in j['results']:
            if o['id'] in seen_obs: continue
            for p in o.get('photos', []):
                lic = (p.get('license_code') or '').lower()
                if lic in LICENSES and p.get('url'):
                    got.append({'id': p['id'], 'url': p['url'].replace('/square.', '/medium.'), 'attribution': p.get('attribution', ''),
                                'license': lic, 'obs': o['id'], 'user': (o.get('user') or {}).get('login', '')})
                    seen_obs.add(o['id']); break
            if len(got) >= PHOTOS_PER_SPECIES: break
        if len(j['results']) < 200: break
        page += 1
    photos[tid] = got
    if n % 20 == 0 or n == len(todo) - 1:
        save('photos.json', photos); print(f'  {n + 1}/{len(todo)}  {species[tid]["common"]}: {len(got)} photos')
save('photos.json', photos)
assert all(p['license'] in LICENSES for ps in photos.values() for p in ps), 'non-commercial photo slipped in'
print('Usable photos:', sum(len(v) for v in photos.values()))

In [ ]:
# ⑥ Download the photos (resized to 256 px, saved to Drive so a restart doesn't re-download) ─────────
IMGDIR = os.path.join(WORK, 'images')
def fetch(tid, p):
    path = os.path.join(IMGDIR, tid, f"{p['id']}.jpg")
    if os.path.exists(path): return True
    for attempt in range(3):
        try:
            r = requests.get(p['url'], headers=UA, timeout=60)
            if r.status_code != 200: time.sleep(2); continue
            im = Image.open(io.BytesIO(r.content)).convert('RGB')
            s = 256 / min(im.size); im = im.resize((max(256, round(im.width * s)), max(256, round(im.height * s))), Image.BICUBIC)
            os.makedirs(os.path.dirname(path), exist_ok=True); im.save(path, quality=90)
            return True
        except Exception: time.sleep(2)
    return False

jobs = [(t, p) for t, ps in photos.items() for p in ps]
print(len(jobs), 'photos…')
ok = 0
with cf.ThreadPoolExecutor(16) as ex:
    for i, done in enumerate(ex.map(lambda a: fetch(*a), jobs)):
        ok += done
        if i % 5000 == 0: print(f'  {i}/{len(jobs)}')
print(ok, 'photos ready')

In [ ]:
# ⑦ Final class list and a train / check split ────────────────────────────────────────
random.seed(SEED)
items = {}
for tid, ps in photos.items():
    have = [p for p in ps if os.path.exists(os.path.join(IMGDIR, tid, f"{p['id']}.jpg"))]
    if len(have) >= MIN_PHOTOS: items[tid] = have
dropped = sorted(set(species) - set(items), key=lambda t: species[t]['common'])
CLASSES = sorted(items, key=lambda t: species[t]['sci'])
print(len(CLASSES), 'species kept;', len(dropped), 'left out for too few CC0/CC BY photos')
missing_birddex = [species[t]['common'] for t in dropped if species[t]['app_id']]
if missing_birddex: print('BirdDex birds left out:', ', '.join(missing_birddex))

# copy to fast local disk
if os.path.exists(FAST): shutil.rmtree(FAST)
train, val = [], []
for ci, tid in enumerate(CLASSES):
    ps = items[tid][:]; random.shuffle(ps)
    k = max(3, len(ps) // 10)
    for j, p in enumerate(ps):
        src = os.path.join(IMGDIR, tid, f"{p['id']}.jpg"); dst = os.path.join(FAST, tid, f"{p['id']}.jpg")
        os.makedirs(os.path.dirname(dst), exist_ok=True); shutil.copyfile(src, dst)
        (val if j < k else train).append((dst, ci))
print(len(train), 'training photos,', len(val), 'check photos')

In [ ]:
# ⑧ Train ────────────────────────────────────────────────────────────────────────────
import timm, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
tf_train = T.Compose([T.RandomResizedCrop(IMG, scale=(0.35, 1.0)), T.RandomHorizontalFlip(), T.ColorJitter(0.2, 0.2, 0.2, 0.02), T.ToTensor(), T.Normalize(MEAN, STD)])
tf_val = T.Compose([T.Resize(256), T.CenterCrop(IMG), T.ToTensor(), T.Normalize(MEAN, STD)])
class DS(Dataset):
    def __init__(s, rows, tf): s.rows, s.tf = rows, tf
    def __len__(s): return len(s.rows)
    def __getitem__(s, i): p, y = s.rows[i]; return s.tf(Image.open(p).convert('RGB')), y
dl_train = DataLoader(DS(train, tf_train), batch_size=96, shuffle=True, num_workers=2, pin_memory=True, drop_last=True, persistent_workers=True)
dl_val = DataLoader(DS(val, tf_val), batch_size=128, num_workers=2, pin_memory=True)

torch.manual_seed(SEED)
model = timm.create_model('efficientnet_b0', pretrained=True, num_classes=len(CLASSES)).cuda()
ckpt_path = os.path.join(WORK, 'checkpoint.pt')
start, best = 0, 0.0
opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.02)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=1e-3, total_steps=EPOCHS * len(dl_train), pct_start=0.1)
scaler = torch.amp.GradScaler()
if os.path.exists(ckpt_path):
    ck = torch.load(ckpt_path, map_location='cuda')
    if ck['classes'] == CLASSES:
        model.load_state_dict(ck['model']); opt.load_state_dict(ck['opt']); sched.load_state_dict(ck['sched']); start, best = ck['epoch'] + 1, ck['best']
        print('Resuming at epoch', start + 1)
lossf = nn.CrossEntropyLoss(label_smoothing=0.1)

def evaluate():
    model.eval(); c1 = c5 = n = 0
    with torch.no_grad(), torch.autocast('cuda', dtype=torch.float16):
        for x, y in dl_val:
            x, y = x.cuda(non_blocking=True), y.cuda(non_blocking=True)
            top = model(x).topk(5, 1).indices
            c1 += (top[:, 0] == y).sum().item(); c5 += (top == y[:, None]).any(1).sum().item(); n += len(y)
    return c1 / n, c5 / n

for ep in range(start, EPOCHS):
    model.train(); t0 = time.time(); tot = 0
    for i, (x, y) in enumerate(dl_train):
        x, y = x.cuda(non_blocking=True), y.cuda(non_blocking=True)
        with torch.autocast('cuda', dtype=torch.float16): loss = lossf(model(x), y)
        opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step(); tot += loss.item()
    top1, top5 = evaluate()
    print(f'epoch {ep + 1}/{EPOCHS}  loss {tot / len(dl_train):.3f}  check top-1 {top1:.1%}  top-5 {top5:.1%}  ({(time.time() - t0) / 60:.1f} min)')
    if top1 >= best: best = top1; torch.save(model.state_dict(), os.path.join(WORK, 'best.pt'))
    torch.save({'model': model.state_dict(), 'opt': opt.state_dict(), 'sched': sched.state_dict(), 'epoch': ep, 'best': best, 'classes': CLASSES}, ckpt_path)
print('Best check accuracy (top-1):', f'{best:.1%}')

In [ ]:
# ⑨ Export for the browser, and check the exported model gives the same answers ─────────────
import numpy as np, onnxruntime as ort
model.load_state_dict(torch.load(os.path.join(WORK, 'best.pt'))); model.eval().cpu()
OUT = '/content/out'; os.makedirs(OUT, exist_ok=True)
onnx_path = os.path.join(OUT, 'birdid.onnx')
dummy = torch.randn(1, 3, IMG, IMG)
try: torch.onnx.export(model, dummy, onnx_path, input_names=['image'], output_names=['logits'], opset_version=17, dynamo=False)
except TypeError: torch.onnx.export(model, dummy, onnx_path, input_names=['image'], output_names=['logits'], opset_version=17)
sess = ort.InferenceSession(onnx_path, providers=['CPUExecutionProvider'])
agree = n = c1 = 0
for x, y in DataLoader(DS(val[:600], tf_val), batch_size=1):
    a = sess.run(None, {'image': x.numpy()})[0][0]
    with torch.no_grad(): b = model(x)[0].numpy()
    agree += int(a.argmax() == b.argmax()); c1 += int(a.argmax() == y.item()); n += 1
print(f'exported model agrees with PyTorch on {agree}/{n}; top-1 on these: {c1 / n:.1%}')
assert agree / n > 0.98, 'export mismatch'
print('model size:', round(os.path.getsize(onnx_path) / 1e6, 1), 'MB')
top1, top5 = best, None

In [ ]:
# ⑩ Labels, credits and settings for the app ──────────────────────────────────────────
def who(attr):  # "(c) Jane Doe, some rights reserved (CC BY)" → "Jane Doe"
    m = re.match(r'\s*(?:\(c\)\s*)?(.*?),\s*(some|no) rights reserved', attr or '', re.I)
    return (m.group(1) if m and m.group(1).strip() else 'Unknown photographer').strip()
labels = [{'i': i, 'inat': int(t), 'sci': species[t]['sci'], 'common': species[t]['common'], 'app': species[t]['app_id']} for i, t in enumerate(CLASSES)]
used = [(t, p) for t in CLASSES for p in items[t]]
people = {}
for t, p in used:
    k = who(p['attribution']); e = people.setdefault(k, {'name': k, 'photos': 0, 'licenses': set()})
    e['photos'] += 1; e['licenses'].add(p['license'])
credits = {
    'source': 'iNaturalist (inaturalist.org) research-grade observations',
    'licenses': {'cc0': 'https://creativecommons.org/publicdomain/zero/1.0/', 'cc-by': 'https://creativecommons.org/licenses/by/4.0/'},
    'note': 'Photos were resized and cropped to train the BirdlyQuest bird ID model. They are not shown in the app.',
    'photographers': sorted(({'name': e['name'], 'photos': e['photos'], 'licenses': sorted(e['licenses'])} for e in people.values()), key=lambda e: e['name'].lower()),
}
with open(os.path.join(OUT, 'credits_photos.csv'), 'w', newline='') as f:
    w = csv.writer(f); w.writerow(['photo_id', 'photo_url', 'observation_url', 'attribution', 'license', 'species'])
    for t, p in used: w.writerow([p['id'], f"https://www.inaturalist.org/photos/{p['id']}", f"https://www.inaturalist.org/observations/{p['obs']}", p['attribution'], p['license'], species[t]['sci']])
json.dump(labels, open(os.path.join(OUT, 'labels.json'), 'w'), ensure_ascii=False)
json.dump(credits, open(os.path.join(OUT, 'credits.json'), 'w'), ensure_ascii=False)
json.dump({'model': 'efficientnet_b0 (timm, ImageNet-1k start) fine-tuned', 'input': IMG, 'resize': 256, 'mean': MEAN, 'std': STD,
           'classes': len(CLASSES), 'check_top1': round(best, 4), 'photos': len(used), 'photographers': len(people),
           'licenses': LICENSES, 'trained': time.strftime('%Y-%m-%d')}, open(os.path.join(OUT, 'config.json'), 'w'), indent=1)
zip_path = os.path.join(WORK, 'birdlyquest-birdid.zip')
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for name in ['birdid.onnx', 'labels.json', 'credits.json', 'credits_photos.csv', 'config.json']: z.write(os.path.join(OUT, name), name)
print(f'{len(CLASSES)} species · {len(used)} photos · {len(people)} photographers · check top-1 {best:.1%}')
print('Saved to Drive:', zip_path, f'({os.path.getsize(zip_path) / 1e6:.1f} MB)')
from google.colab import files
files.download(zip_path)